# Backend-7 (추가 백엔드-7 설계) — Total 조립 불량 손실 시간

본 노트북은 설계서(2026-01-30_추가_백엔드-7_설계.pdf) 기준으로 아래를 계산합니다.

- 입력: `prod_day(YYYYMMDD)`, `shift_type(day|night)`
- 손실 시간: `d1_machine_log.afa_fail_wasted_time`의 `from_time ~ to_time` 구간을
  **주/야간 생산 시간대에 맞춰 분할**하여 초 단위로 집계
- 조립 공정 불량 제품 개수: 집계에 사용된 **데이터 행 수**
  - 단, 주/야간 시간이 겹치는 행은 **더 많은 손실 시간을 가진 shift에만 1개**로 카운트
- 재작업 시간: `e1_FCT_ct.fct_whole_op_ct`에서
  `month = (window 기준) 이전달`, `station='whole'`, `remark='PD'`의 `final_ct`를 가져와
  `final_ct * 조립 공정 불량 제품 개수`
- Total 조립 불량 손실 시간 = (손실 시간 + 재작업 시간) / 3600  → **시간 단위**
- `updated_at`: dataframe 생성 시각(timestamp)

## 시간 규칙(문서 그대로)
- 주간(day): [D-DAY] 08:30:00 ~ 20:29:59 (양끝 포함)
- 야간(night): [D-DAY] 20:30:00 ~ 23:59:59 + [D+1] 00:00:00 ~ 08:29:59 (양끝 포함)

## 초 계산 규칙(문서 예시 일치)
- 이벤트 구간은 **(from_time, to_time]** 로 해석 (시작은 제외, 종료는 포함)
  - 예: from=08:28:59, to=08:30:00 → 총 61초
  - 분할: night=60초(08:29:00~08:29:59), day=1초(08:30:00)


In [1]:
# =========================
# 0) 설치(필요 시)
# =========================
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

from __future__ import annotations

from decimal import Decimal, ROUND_HALF_UP
from datetime import datetime, date, timedelta, time as dtime
from zoneinfo import ZoneInfo

import pandas as pd
from sqlalchemy import create_engine, text

KST = ZoneInfo("Asia/Seoul")

In [2]:
# =========================
# 1) DB 설정 (문서 사양)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

AFA_SCHEMA = "d1_machine_log"
AFA_TABLE  = "afa_fail_wasted_time"

CT_SCHEMA  = "e1_FCT_ct"   # FCT는 대문자
CT_TABLE   = "fct_whole_op_ct"

In [3]:
# =========================
# 2) DB 엔진 (노트북용: pool 최소)
# =========================
def make_engine():
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

engine = make_engine()

In [4]:
# =========================
# 3) 시간 유틸
# =========================
DAY_START   = dtime(8, 30, 0)
DAY_END     = dtime(20, 29, 59)
NIGHT_START = dtime(20, 30, 0)
NIGHT_END   = dtime(8, 29, 59)

def _parse_yyyymmdd(s: str) -> date:
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def _fmt_yyyymmdd(d: date) -> str:
    return d.strftime("%Y%m%d")

def round_time_str_to_hhmmss(t: str) -> str:
    # 예: "06:41:23.21" -> "06:41:23" (0.5에서 반올림, half-up)
    t = (t or "").strip()
    if not t:
        return "00:00:00"
    hh, mm, ss = t.split(":")
    sec = Decimal(ss)
    total = Decimal(int(hh)) * 3600 + Decimal(int(mm)) * 60 + sec
    total_rounded = int(total.quantize(Decimal("1"), rounding=ROUND_HALF_UP))
    total_rounded = max(0, total_rounded)

    hh2 = (total_rounded // 3600) % 24
    mm2 = (total_rounded % 3600) // 60
    ss2 = total_rounded % 60
    return f"{hh2:02d}:{mm2:02d}:{ss2:02d}"

def build_dt(end_day_yyyymmdd: str, time_text: str) -> datetime:
    d = _parse_yyyymmdd(end_day_yyyymmdd)
    hhmmss = round_time_str_to_hhmmss(time_text)
    hh, mm, ss = map(int, hhmmss.split(":"))
    return datetime(d.year, d.month, d.day, hh, mm, ss, tzinfo=KST)

def shift_window(prod_day: str, shift_type: str) -> tuple[datetime, datetime]:
    d = _parse_yyyymmdd(prod_day)
    if shift_type == "day":
        ws = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
        we = datetime(d.year, d.month, d.day, 20, 29, 59, tzinfo=KST)
        return ws, we
    if shift_type == "night":
        ws = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
        we = datetime(d.year, d.month, d.day, 8, 29, 59, tzinfo=KST) + timedelta(days=1)
        return ws, we
    raise ValueError("shift_type must be 'day' or 'night'")

def prev_month_yyyymm_by_window(prod_day: str, shift_type: str) -> str:
    ws, _ = shift_window(prod_day, shift_type)
    first = date(ws.year, ws.month, 1)
    prev_last = first - timedelta(days=1)
    return prev_last.strftime("%Y%m")

def _epoch_sec(dt: datetime) -> int:
    # tz-aware dt -> epoch seconds (정수)
    return int(dt.timestamp())

In [5]:
# =========================
# 4) 이벤트 구간 (from, to] 기준 overlap(초)
#    - 이벤트: (from_dt, to_dt]  (start 제외, end 포함)
#    - 윈도우: [ws, we]          (양끝 포함)
# =========================
def overlap_seconds_event(from_dt: datetime, to_dt: datetime, ws: datetime, we: datetime) -> int:
    # (from_dt, to_dt] ∩ [ws, we] 의 초 개수 (정수 초 라벨 기준)
    s = _epoch_sec(from_dt)
    e = _epoch_sec(to_dt)
    a = _epoch_sec(ws)
    b = _epoch_sec(we)

    # 이벤트가 기여하는 초 라벨: (s, e] = [s+1, e]
    start = max(s + 1, a)
    end   = min(e, b)
    if end < start:
        return 0
    return end - start + 1

def allocate_seconds_by_prod_shift(from_dt: datetime, to_dt: datetime, prod_day: str, shift_type: str) -> int:
    ws, we = shift_window(prod_day, shift_type)
    return overlap_seconds_event(from_dt, to_dt, ws, we)

In [6]:
# =========================
# 5) DB 로드 (+ CT 테이블 대소문자/스키마 자동해결)
# =========================
from sqlalchemy import text

def load_afa_rows(engine, day_from: str, day_to: str) -> pd.DataFrame:
    sql = text(f'''
        SELECT end_day, from_time, to_time
        FROM {AFA_SCHEMA}.{AFA_TABLE}
        WHERE end_day >= :dfrom AND end_day <= :dto
    ''')
    return pd.read_sql(sql, engine, params={"dfrom": day_from, "dto": day_to})

def _quote_ident(name: str) -> str:
    # Postgres: 대문자/특수문자 있으면 " "로 감싸야 원문 그대로 인식됨
    if any(c.isupper() for c in name) or any(c in name for c in ['-', ' ', '.']):
        return '"' + name.replace('"', '""') + '"'
    return name

def _find_table_case_insensitive(engine, want_schema: str, want_table: str):
    # 1) schema+table 둘 다 대소문자 무시로 탐색
    sql = text('''
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE lower(table_name) = lower(:tbl)
          AND lower(table_schema) = lower(:sch)
        LIMIT 1
    ''')
    with engine.connect() as conn:
        row = conn.execute(sql, {"tbl": want_table, "sch": want_schema}).fetchone()
        if row:
            return row[0], row[1]

        # 2) 스키마가 다를 수도 있으니 table_name 기준으로 전체 탐색(1건)
        sql2 = text('''
            SELECT table_schema, table_name
            FROM information_schema.tables
            WHERE lower(table_name) = lower(:tbl)
            ORDER BY table_schema
            LIMIT 1
        ''')
        row2 = conn.execute(sql2, {"tbl": want_table}).fetchone()
        return (row2[0], row2[1]) if row2 else (None, None)

# --- CT 테이블 실제 위치/대소문자 자동 해결 ---
FOUND_SCHEMA, FOUND_TABLE = _find_table_case_insensitive(engine, CT_SCHEMA, CT_TABLE)

if not FOUND_SCHEMA:
    print(f"[ERROR] CT source table not found: table={CT_TABLE} (schema hint={CT_SCHEMA})")
    CT_FQN = None
else:
    CT_SCHEMA = FOUND_SCHEMA
    CT_TABLE = FOUND_TABLE
    CT_FQN = f"{_quote_ident(CT_SCHEMA)}.{_quote_ident(CT_TABLE)}"
    print(f"[OK] CT source resolved -> {CT_FQN}")

def load_pd_final_ct(engine, prev_month_yyyymm: str) -> float | None:
    """
    CT_FQN에서 month=이전달, station='whole', remark='PD' 의 final_ct 1건 반환.
    """
    if not CT_FQN:
        return None

    chk = text('''
        SELECT 1
        FROM information_schema.columns
        WHERE table_schema = :sch AND table_name = :tbl AND column_name = 'updated_at'
        LIMIT 1
    ''')
    with engine.connect() as conn:
        has_updated = conn.execute(chk, {"sch": CT_SCHEMA, "tbl": CT_TABLE}).fetchone() is not None

        if has_updated:
            sql = text(f'''
                SELECT final_ct
                FROM {CT_FQN}
                WHERE month = :m AND station = 'whole' AND remark = 'PD'
                ORDER BY updated_at DESC
                LIMIT 1
            ''')
        else:
            sql = text(f'''
                SELECT final_ct
                FROM {CT_FQN}
                WHERE month = :m AND station = 'whole' AND remark = 'PD'
                LIMIT 1
            ''')

        row = conn.execute(sql, {"m": prev_month_yyyymm}).fetchone()

    if not row:
        return None
    try:
        return float(row[0])
    except Exception:
        return None

[OK] CT source resolved -> "e1_FCT_ct".fct_whole_op_ct


In [11]:
# =========================
# (추가) 초 -> "h시간 m분 s초" 포맷 유틸
# =========================
def sec_to_hms_text(total_sec: float) -> str:
    s = int(round(total_sec))
    if s <= 0:
        return "0초"

    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60

    parts = []
    if h > 0:
        parts.append(f"{h}시간")
    if m > 0:
        parts.append(f"{m}분")
    if ss > 0:
        parts.append(f"{ss}초")

    return " ".join(parts) if parts else "0초"

# =========================
# 6) Backend-7 계산 (문서 사양)
# =========================
def compute_backend7(engine, prod_day: str, shift_type: str) -> pd.DataFrame:
    ws, we = shift_window(prod_day, shift_type)
    updated_at = datetime.now(tz=KST)

    # end_day는 ws~we에 걸쳐 prod_day 또는 prod_day+1이 될 수 있음.
    # 안전하게 prod_day-1 ~ prod_day+1 로드 후 Python에서 필터.
    d = _parse_yyyymmdd(prod_day)
    day_from = _fmt_yyyymmdd(d - timedelta(days=1))
    day_to   = _fmt_yyyymmdd(d + timedelta(days=1))

    df = load_afa_rows(engine, day_from, day_to)
    if df.empty:
        return pd.DataFrame([{
            "prod_day": prod_day,
            "shift_type": shift_type,
            "Total 조립 불량 손실 시간(hours)": 0.0,
            "Total 조립 불량 손실 시간(HMS)": "0시간 0분 0초",
            "updated_at": updated_at,
        }])

    wasted_sec = 0
    defect_cnt = 0

    for r in df.itertuples(index=False):
        end_day = str(r.end_day)
        from_dt = build_dt(end_day, str(r.from_time))
        to_dt   = build_dt(end_day, str(r.to_time))
        if to_dt < from_dt:
            to_dt = to_dt + timedelta(days=1)

        # (from,to] 기준으로 윈도우 교집합이 없으면 skip
        if to_dt <= ws or from_dt >= we:
            continue

        sec_this = overlap_seconds_event(from_dt, to_dt, ws, we)
        if sec_this <= 0:
            continue

        wasted_sec += sec_this

        # 불량 제품 개수 카운트 규칙:
        # - 같은 행이 day/night 모두에 걸치면 더 큰 쪽으로 1개 귀속
        day_sec = allocate_seconds_by_prod_shift(from_dt, to_dt, prod_day, "day")
        night_sec = allocate_seconds_by_prod_shift(from_dt, to_dt, prod_day, "night")

        if shift_type == "day":
            if day_sec >= night_sec and day_sec > 0:
                defect_cnt += 1
        else:
            if night_sec > day_sec and night_sec > 0:
                defect_cnt += 1

    prev_m = prev_month_yyyymm_by_window(prod_day, shift_type)
    final_ct = load_pd_final_ct(engine, prev_m)
    rework_sec = (final_ct * defect_cnt) if (final_ct is not None) else 0.0

    total_sec = float(wasted_sec) + float(rework_sec)
    total_loss_hours = total_sec / 3600.0
    total_loss_hms = sec_to_hms_text(total_sec)

    return pd.DataFrame([{
        "prod_day": prod_day,
        "shift_type": shift_type,
        "Total 조립 불량 손실 시간(hours)": total_loss_hours,
        "Total 조립 불량 손실 시간(HMS)": total_loss_hms,
        "updated_at": updated_at,
    }])

In [12]:
# =========================
# 7) 실행 예시: 20260124 야간
# =========================
prod_day = "20260124"
shift_type = "night"

result_df = compute_backend7(engine, prod_day, shift_type)
result_df

,prod_day,shift_type,Total 조립 불량 손실 시간(hours),Total 조립 불량 손실 시간(HMS),updated_at
0,20260124,night,0.011339,41초,2026-01-30 11:04:53.227877+09:00
